# Benchmarking KGs 

Everything you need to benchmark knowledge-graph `Pipelines`, as briefly as possible.

The five stage benchmarks — `Dedup`, `Chunking`, `Extraction`, `Resolution`, `Quality` — each isolate ONE pipeline stage and score it against the bundled benchmarking gold data. Every run returns a `StageResult` — just `print()` it to see a readable per-pipeline report.


In [ ]:
# ── Setup: imports ─────────────────────────────────────────────
from kglab.benchmark_pipeline import Benchmark
from kglab.pipelines import Baseline

### 1) Dedup — duplicate detection

Scores how well the pipeline spots duplicate / near-duplicate document pairs against the bundled DBLP-ACM gold (full gold, no cap). Returns a `StageResult` — `print()` renders a readable per-pipeline table (best marked `*`).


In [ ]:
dedup = Benchmark.Dedup().run(
    pipelines={
        "surface": Baseline(),
    }
)

print(dedup)  # dedup: <class 'kglab.benchmark_pipeline.report.StageResult'>


Dedup — near-duplicate detection (DBLP-ACM gold)   [dataset: dedup_gold.jsonl]
pipeline | precision (TP/(TP+FP)) | recall (TP/(TP+FN)) | f1 (2·TP/(2·TP+FP+FN)) | accuracy ((TP+TN)/N) | threshold | runtime_s
---------+------------------------+---------------------+------------------------+----------------------+-----------+----------
minhash  |                  1.000 |               0.229 |                0.373 * |                0.615 |     0.850 |     3.709
  * = best f1
  TP = true positives · FP = false positives · FN = false negatives · TN = true negatives
  Higher F1 = better duplicate detection. Case-sensitive methods score low recall on DBLP-ACM (the duplicates differ in case) — an honest finding, not a bug.



### 2) Chunking — chunk boundaries vs gold entities

Does chunking keep gold entities intact? Scores chunk-boundary F1 against the bundled gold. First 200 records to keep the demo fast.


In [ ]:
# The chunking benchmark scores sentence chunking by default (offline-safe).
chunking = Benchmark.Chunking().run(
    pipelines={
        "sentence": Baseline(),
    },
    max_records=200,
)
print(chunking)


Chunking — chunk boundaries keep gold entities intact   [dataset: chunking_gold.jsonl]
pipeline | precision (TP/(TP+FP)) | recall (TP/(TP+FN)) | f1 (2·TP/(2·TP+FP+FN)) | n_samples | runtime_s
---------+------------------------+---------------------+------------------------+-----------+----------
sentence |                  1.000 |               1.000 |                1.000 * |       200 |     0.002
  * = best f1
  TP = true positives · FP = false positives · FN = false negatives · TN = true negatives
  Higher F1 = chunks less often cut through a gold entity; 1.0 means every gold entity survives in one chunk.



### 3) Extraction — NER spans vs gold entities

How well does spaCy NER match the gold entity spans? Scores span-level F1 against the bundled CoNLL-2003 gold (first 500 records). Uses `en_core_web_sm` — swap in the default `en_core_web_lg` for a stronger model (needs `python -m spacy download en_core_web_lg`).


In [16]:
extraction = Benchmark.Extraction().run(
    pipelines={
        "spacy": Baseline(entity_options={"model_name": "en_core_web_sm"}),
    },
    max_records=500,
)
print(extraction)


Extraction — NER spans vs gold entities   [dataset: ner_gold.jsonl]
pipeline | precision (TP/(TP+FP)) | recall (TP/(TP+FN)) | f1 (2·TP/(2·TP+FP+FN)) | type_accuracy | n_scored | runtime_s
---------+------------------------+---------------------+------------------------+---------------+----------+----------
spacy    |                  0.862 |               0.731 |                0.791 * |         0.894 |      802 |     2.134
  * = best f1
  TP = true positives · FP = false positives · FN = false negatives · TN = true negatives
  Higher F1 = extracted NER spans match the gold annotations. Only labels in the chosen gold's schema are scored (n_scored shows how many predictions counted); type_accuracy is how often the type is also correct. Pick the gold that fits your extractor — print(Benchmark.Extraction.golds()).



### 4) Resolution — entity merging (cluster F1)

How well are name variants merged into the gold clusters? Full bundled gold — a smoke test (2 hand-written clusters, so the number carries no real meaning; real gold is license-gated).


In [17]:
resolution = Benchmark.Resolution().run(
    pipelines={
        "string": Baseline(),
    }
)
print(resolution)


Resolution — entity merging (cluster F1)   [dataset: resolution_gold.jsonl]
pipeline | precision | recall |      f1 | pairwise_f1 (2·TP/(2·TP+FP+FN)) | threshold | runtime_s
---------+-----------+--------+---------+---------------------------------+-----------+----------
string   |     1.000 |  0.400 | 0.571 * |                           0.000 |     0.850 |     0.000
  * = best f1
  TP = true positives · FP = false positives · FN = false negatives · TN = true negatives
  Higher F1 = mentions are merged the way the gold clusters group them. Smoke test only: the bundled gold is 2 hand-written clusters, so this number is not a real measurement — supply real gold via dataset= or --dataset.



### 5) Quality — keep/reject filter accuracy

How well does the quality filter agree with the gold keep/reject labels? Full bundled gold — a smoke test (9 hand-written records, so the number carries no real meaning; real gold is license-gated).


In [18]:
quality = Benchmark.Quality().run(
    pipelines={
        "surface": Baseline(),
    }
)
print(quality)


Quality — keep/reject filter vs gold labels   [dataset: quality_gold.jsonl]
pipeline | accuracy ((TP+TN)/N) | precision (TP/(TP+FP)) | recall (TP/(TP+FN)) | f1 (2·TP/(2·TP+FP+FN)) | runtime_s
---------+----------------------+------------------------+---------------------+------------------------+----------
surface  |              1.000 * |                  1.000 |               1.000 |                  1.000 |     0.000
  * = best accuracy
  TP = true positives · FP = false positives · FN = false negatives · TN = true negatives
  Higher accuracy = the filter agrees with the gold keep/reject labels. Smoke test only: the bundled gold is 9 hand-written records, so this number is not a real measurement — supply real gold via dataset= or --dataset.

